In [ ]:
import os
from matplotlib import pyplot as plt
import numpy as np
import sklearn
import sklearn.model_selection
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import seaborn as sns


In [ ]:

def display_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)

    plt.rcParams["figure.figsize"] = [12,9]
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.show()


In [ ]:
from obspy.core import read
import obspy

data_path = 'data'

class1_data_path = os.path.join(data_path, 'class1')
class2_data_path = os.path.join(data_path, 'class2')
class3_data_path = os.path.join(data_path, 'class3')


fix_size = 32_000

def load_file_data(file_data_path):
    
    if not os.path.isfile(file_data_path):
        return None
    
    data = np.array([])
    # with open(file_data_path, 'r') as data_file:
    #     data = np.loadtxt(data_file, delimiter=' ')

    singlechannel = obspy.read(file_data_path)
    data = np.array(singlechannel[0].data)


    if data.shape[0] > fix_size:
        data = data[:fix_size, :]
    else:
        x = fix_size // data.shape[0]
        for i in range(x):
            data = np.concatenate((data, data), axis=0)

    data = data[:fix_size]
    return data


def load_data(folder_data_path):

    result_data = np.array([])
    for data_file_name in os.listdir(folder_data_path):
        
        data_file_path = os.path.join(folder_data_path, data_file_name)
        data = load_file_data(data_file_path)
        result_data = np.append(result_data, data)

    x = len(result_data) // fix_size
    result_data = result_data.reshape((x, fix_size))
    return result_data

class1_data = load_data(class1_data_path)
class1_label = np.ones((class1_data.shape[0]))

class2_data = load_data(class2_data_path)
class2_data = np.split(class2_data,4)[0]

class2_label = np.zeros((class2_data.shape[0]))

class3_data = load_data(class3_data_path)
class3_label = np.ones((class3_data.shape[0])) + 1

print(class1_data.shape)
print(class2_data.shape)
print(class3_data.shape)

data = np.concatenate((class1_data, class2_data, class3_data), axis=0)
print(data.shape)
label = np.concatenate((class1_label, class2_label, class3_label), axis=0)
print(label.shape)

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(data, label, test_size=0.35, random_state=42)

print(X_train.shape)
print(X_test.shape)


In [ ]:

# Path: model.ipynb
from sklearn import svm
from sklearn.metrics import accuracy_score

clf = svm.SVC()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))


In [ ]:
# apply linear kernel
clf = svm.SVC(kernel='linear')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['class1', 'class2'])
disp.plot()




In [ ]:
# Use CNN to train model
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt


# Path: model.ipynb
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(X_train.shape)
print(X_test.shape)

model = models.Sequential(
    [
        layers.Conv1D(256, 3, activation="relu", input_shape=(X_train.shape[1], 1)),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 3, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, activation="relu"),
        layers.Dropout(0.5),
        layers.MaxPooling1D(2),
        layers.Conv1D(16, 3, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)


model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(X_train, y_train, epochs=15, validation_data=(X_test, y_test))

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()


In [ ]:
model.summary()
model.save('model.h5')

In [ ]:
# draw confusion matrix
y_pred = model.predict(X_test)
y_pred = np.where(y_pred > 0.5, 1, 0)
print(accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['class2', 'class1'])
disp.plot()

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=3)

# decrease dimension
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1]))

neigh.fit(X_train, y_train)


# draw confusion matrix
y_pred = neigh.predict(X_test)
print(accuracy_score(y_test, y_pred))

display_confusion_matrix(y_test, y_pred)


In [ ]:
# random forest
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(max_depth=13, random_state=0)
clf.fit(X_train, y_train)

# draw confusion matrix
y_pred = clf.predict(X_test)

print(accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['class3', 'class2', 'class1'], columns=[ 'class3' ,'class2', 'class1'])
plt.figure(figsize=(5.5,4))
sns.heatmap(cm_df, annot=True)
plt.title('Random Forest \nAccuracy:{0:.3f}'.format(accuracy_score(y_test, y_pred)))
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

